In [1]:
!pip install google-api-python-client pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 1.4 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [google-api-python-client]32m8/9 [google-api-python-client]


In [54]:
!pip install deep-translator


In [58]:
from googleapiclient.discovery import build
import pandas as pd
import re

# 1️⃣ Replace with your YouTube Data API key
API_KEY = "YOUR-YOUTUBE-DATA-API-KEY"

# 2️⃣ Extract video ID from the link
def extract_video_id(url):
    pattern = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
    match = re.search(pattern, url)
    return match.group(1) if match else None

video_url = "https://www.youtube.com/watch?v=aRePC8Fp-YI"  # Example video
video_id = extract_video_id(video_url)

# 3️⃣ Initialize YouTube API client
youtube = build("youtube", "v3", developerKey=API_KEY)

# 4️⃣ Fetch comments (fetch as many pages as you want)
comments = []
next_page_token = None
total_fetched = 0
max_comments_to_fetch = 1000  # adjust for your quota

while True:
    response = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText",
        pageToken=next_page_token
    ).execute()

    for item in response["items"]:
        comment = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "comment": comment.get("textDisplay"),
            "like_count": comment.get("likeCount", 0),
        })

    total_fetched += len(response["items"])
    next_page_token = response.get("nextPageToken")

    # Stop if reached your fetch limit
    if not next_page_token or total_fetched >= max_comments_to_fetch:
        break

print(f"✅ Fetched total {len(comments)} comments.")

# 5️⃣ Convert to DataFrame
df = pd.DataFrame(comments)

# 6️⃣ Sort by like count and keep top 100
df_sorted = df.sort_values(by="like_count", ascending=False).head(100).reset_index(drop=True)

# 7️⃣ Save to CSV
output_file = "youtube_top100_liked_comments.csv"
df_sorted.to_csv(output_file, index=False)

print(f"🏆 Saved top {len(df_sorted)} liked comments to '{output_file}'")
print("\nSample top comments:\n")
print(df_sorted[['like_count', 'comment']].head(5))


✅ Fetched total 1000 comments.
🏆 Saved top 100 liked comments to 'youtube_top100_liked_comments.csv'

Sample top comments:

   like_count                                            comment
0         912  the unique concept and the vintage vibes, we a...
1         290  Anurag Kashyap and Manoj Bajpayee ❤\n\nNo wond...
2          72  Now that's what i called cinema of realism! we...
3          21  Manoj Bajpai is hands down the most versatile ...
4          15  It was such an honour to write the music for t...


In [52]:
import pandas as pd
import joblib
import re
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are already available locally
try:
    stop_words = set(stopwords.words('english'))
except:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

# 1️⃣ Load the pre-trained sentiment model
model_path = "binary_sentiment_model.pkl"  # path to your trained model
pipeline = joblib.load(model_path)

# 2️⃣ Load the YouTube comments dataset
file_path = "youtube_top100_liked_comments.csv"  # your file
comments_df = pd.read_csv(file_path)

# 3️⃣ Clean comment text (same logic as training)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)            # remove URLs
    text = re.sub(r'[^a-z0-9\s]', ' ', text)       # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

comments_df['cleaned_comment'] = comments_df['comment'].apply(clean_text)

# 4️⃣ Predict sentiment
preds = pipeline.predict(comments_df['cleaned_comment'])
comments_df['sentiment'] = ['positive' if p == 1 else 'negative' for p in preds]

# 5️⃣ (Optional) Add probability scores
probs = pipeline.predict_proba(comments_df['cleaned_comment'])
comments_df['positive_score'] = probs[:, 1]
comments_df['negative_score'] = probs[:, 0]

# 6️⃣ Save to new CSV
output_path = "youtube_comments_with_sentiment.csv"
comments_df.to_csv(output_path, index=False)

print(f"✅ Sentiment analysis completed! Results saved to '{output_path}'")
print(comments_df[['comment', 'sentiment', 'positive_score', 'negative_score']].head(10))


✅ Sentiment analysis completed! Results saved to 'youtube_comments_with_sentiment.csv'
                                             comment sentiment  \
0  1:48. Amrita was right. 😮. She deserves a meda...  negative   
1                When you eat today, Thank a farmar😔  positive   
2  You won't believe it, but the movie Jolly LLB ...  negative   
3  One of the best Franchise ever " Aaj ki genera...  positive   
4  Good to see Arshad back… remembering his dialo...  negative   
5  Watched this movie today \nI request everyone ...  negative   
6  Both Jollys in the house, this looks like doub...  negative   
7  everything is temporary \ntakla judge is perma...  positive   
8  My opinion:- akshay kumar glt aadmi ko support...  positive   
9                       Team ARSHAD (ORIGINAL JOLLY)  positive   

   positive_score  negative_score  
0        0.471642        0.528358  
1        0.605109        0.394891  
2        0.487289        0.512711  
3        0.624667        0.375333  
4     

In [53]:
import pandas as pd

# Load your analyzed dataset
file_path = "youtube_comments_with_sentiment.csv"
df = pd.read_csv(file_path)

# Count number of positive and negative comments
sentiment_counts = df["sentiment"].value_counts()

# Print results
print("🎬 YouTube Trailer Sentiment Summary:")
print(f"Positive comments: {sentiment_counts.get('positive', 0)}")
print(f"Negative comments: {sentiment_counts.get('negative', 0)}")

# Optional: show percentage breakdown
total = len(df)
pos_pct = (sentiment_counts.get('positive', 0) / total) * 100
neg_pct = (sentiment_counts.get('negative', 0) / total) * 100

print(f"\n📊 Positive: {pos_pct:.2f}% | Negative: {neg_pct:.2f}%")


🎬 YouTube Trailer Sentiment Summary:
Positive comments: 59
Negative comments: 41

📊 Positive: 59.00% | Negative: 41.00%


In [55]:
import pandas as pd
import joblib
import re
import nltk
from nltk.corpus import stopwords
from deep_translator import GoogleTranslator

# Download stopwords
try:
    stop_words = set(stopwords.words('english'))
except:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

# 1️⃣ Load your sentiment model
model_path = "binary_sentiment_model.pkl"
pipeline = joblib.load(model_path)

# 2️⃣ Load YouTube comments dataset
file_path = "youtube_top100_liked_comments.csv"
comments_df = pd.read_csv(file_path)

# 3️⃣ Translate comments to English
def translate_to_english(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(str(text))
    except Exception:
        return str(text)  # fallback if translation fails

comments_df["translated_comment"] = comments_df["comment"].apply(translate_to_english)

# 4️⃣ Clean text (same as training)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)            # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)       # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

comments_df["cleaned_comment"] = comments_df["translated_comment"].apply(clean_text)

# 5️⃣ Predict sentiment
preds = pipeline.predict(comments_df["cleaned_comment"])
comments_df["sentiment"] = ["positive" if p == 1 else "negative" for p in preds]

# 6️⃣ Add probability scores
probs = pipeline.predict_proba(comments_df["cleaned_comment"])
comments_df["positive_score"] = probs[:, 1]
comments_df["negative_score"] = probs[:, 0]

# 7️⃣ Save to new CSV
output_path = "youtube_comments_translated_with_sentiment.csv"
comments_df.to_csv(output_path, index=False)

print(f"✅ All comments translated to English and analyzed! Saved to '{output_path}'")
print(comments_df[["comment", "translated_comment", "sentiment"]].head(10))


✅ All comments translated to English and analyzed! Saved to 'youtube_comments_translated_with_sentiment.csv'
                                             comment  \
0  1:48. Amrita was right. 😮. She deserves a meda...   
1                When you eat today, Thank a farmar😔   
2  You won't believe it, but the movie Jolly LLB ...   
3  One of the best Franchise ever " Aaj ki genera...   
4  Good to see Arshad back… remembering his dialo...   
5  Watched this movie today \nI request everyone ...   
6  Both Jollys in the house, this looks like doub...   
7  everything is temporary \ntakla judge is perma...   
8  My opinion:- akshay kumar glt aadmi ko support...   
9                       Team ARSHAD (ORIGINAL JOLLY)   

                                  translated_comment sentiment  
0  1:48. Amrita was right. 😮. She deserves a meda...  negative  
1                When you eat today, Thank a farmar😔  positive  
2  You won't believe it, but the movie Jolly LLB ...  negative  
3  One of the 

In [56]:
import pandas as pd

# Load your analyzed dataset
file_path = "youtube_comments_translated_with_sentiment.csv"
df = pd.read_csv(file_path)

# Count number of positive and negative comments
sentiment_counts = df["sentiment"].value_counts()

# Print results
print("🎬 YouTube Trailer Sentiment Summary:")
print(f"Positive comments: {sentiment_counts.get('positive', 0)}")
print(f"Negative comments: {sentiment_counts.get('negative', 0)}")

# Optional: show percentage breakdown
total = len(df)
pos_pct = (sentiment_counts.get('positive', 0) / total) * 100
neg_pct = (sentiment_counts.get('negative', 0) / total) * 100

print(f"\n📊 Positive: {pos_pct:.2f}% | Negative: {neg_pct:.2f}%")


🎬 YouTube Trailer Sentiment Summary:
Positive comments: 57
Negative comments: 43

📊 Positive: 57.00% | Negative: 43.00%


In [57]:
import pandas as pd

# Load your analyzed dataset
file_path = "youtube_comments_translated_with_sentiment.csv"
df = pd.read_csv(file_path)

# Ensure the columns exist
if "positive_score" in df.columns and "negative_score" in df.columns:
    avg_positive = df["positive_score"].mean()
    avg_negative = df["negative_score"].mean()

    print("🎬 Average Sentiment Scores:")
    print(f"✅ Average Positive Score: {avg_positive:.4f}")
    print(f"❌ Average Negative Score: {avg_negative:.4f}")
else:
    print("⚠️ Columns 'positive_score' and 'negative_score' not found in dataset!")


🎬 Average Sentiment Scores:
✅ Average Positive Score: 0.5126
❌ Average Negative Score: 0.4874
